<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/form-8k/download-press-releases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download Press Releases with Financial Results in Exhibit 99 from SEC 8-K Filings

This tutorial outlines the steps to download press releases (PRs) that disclose annual and quarterly financial results, filed as Exhibit 99 within SEC 8-K filings.

Steps for downloading PRs with financial results in Exhibit 99:

1. Use the Query API to locate all 8-K filings that include press releases with financial results in Exhibit 99.
2. Extract the URLs for each press release from the filing metadata and save them locally.
3. Download the press releases using the list of URLs with the Download API.


In [ ]:
!pip install -q sec-api

In [ ]:
from sec_api import QueryApi

api_key = "YOUR_API_KEY"

queryApi = QueryApi(api_key)

### Exhibit 99 in 8-K Filings

Press releases (PRs) are typically attached to 8-K filings as Exhibit 99. To identify all relevant 8-K filings containing PRs, it is necessary to filter for those that include Exhibit 99, excluding 8-Ks without it.

The example below shows the filing details page from an [8-K filing by Nvidia](https://www.sec.gov/Archives/edgar/data/1045810/000104581023000014/0001045810-23-000014-index.htm), which contains two Exhibit 99 files:

1. Press release with financial results: [_Q4FY23 PRESS RELEASE_ (q4fy23pr.htm)](https://www.sec.gov/Archives/edgar/data/1045810/000104581023000014/q4fy23pr.htm)
2. CFO commentary: [_Q4FY23 CFO COMMENTARY_ (q4fy23cfocommentary.htm)](https://www.sec.gov/Archives/edgar/data/1045810/000104581023000014/q4fy23cfocommentary.htm)

![press-release-in-8-k-filing](https://i.imgur.com/nTdm9NFl.png)

#### Types of Exhibit 99 Content in 8-K Filings

Exhibit 99 in 8-K filings includes a diverse range of material information, often beyond just press releases. Below is an overview of the types of content typically disclosed in Exhibit 99:

- Press releases about annual and quarterly financial results, management changes, and other material events. [Example](https://www.sec.gov/Archives/edgar/data/1699039/000169903923000011/rngr-123122ex991earningsre.htm).
- CFO commentary on company performance. [Example](https://www.sec.gov/Archives/edgar/data/1045810/000104581023000014/q4fy23cfocommentary.htm)
- Investor updates, such as [slides presented at a conference with investors and analysts](https://www.sec.gov/Archives/edgar/data/1324272/000143774923005591/ex_483568.htm), [health conference presentations](https://www.sec.gov/Archives/edgar/data/1564406/000156440623000004/oshjpmconf1092023vfinal.htm), and [general business updates](https://www.sec.gov/Archives/edgar/data/1822691/000095017023006373/grna-ex99_1.htm).
- Announcements of
  - Cash dividends. [Example](https://www.sec.gov/Archives/edgar/data/1046102/000110465923029319/tm238705d1_ex99-1.htm).
  - Share repurchase agreements and authorizations. [Example](https://www.sec.gov/Archives/edgar/data/1047862/000119312523062131/d472749dex99.htm).
  - Agreement and plan of a merger and acquisition. [Example](https://www.sec.gov/Archives/edgar/data/1832950/000149315223006884/ex99-1.htm), [example](https://www.sec.gov/Archives/edgar/data/98222/000143774923005577/ex_484947.htm).
  - Private offering of convertible senior notes. [Example](https://www.sec.gov/Archives/edgar/data/1560385/000110465923029307/tm238677d1_ex99-1.htm).
  - Results of preclinical studies. [Example](https://www.sec.gov/Archives/edgar/data/1430306/000138713123002814/ex99-01.htm).
  - Phase 1 clinicial trial updates. [Example](https://www.sec.gov/Archives/edgar/data/1213809/000143774923005594/ex_484707.htm).
  - Launch of clinical trials. [Example](https://www.sec.gov/Archives/edgar/data/1760903/000149315223006887/ex99-1.htm).
  - New Drug Applications (NDAs). [Example](https://www.sec.gov/Archives/edgar/data/1467154/000146715423000010/a3072023exhibit991.htm).
  - Letter of intent counterparty. [Example](https://www.sec.gov/Archives/edgar/data/1812234/000095014223000623/eh230336657_8k-ex9901.htm).
  - Changes in management, such as appointment of new CFO or board member. [Example](https://www.sec.gov/Archives/edgar/data/1858327/000119312523062162/d457086dex991.htm).
  - Receipt of NASDAQ notification about non-compliance with minimum bid price requirement. [Example](https://www.sec.gov/Archives/edgar/data/1401395/000095017023000208/nept-ex99_1.htm).
  - Receipt of Continued Listing Standard Notice from NYSE. [Example](https://www.sec.gov/Archives/edgar/data/1418100/000141810022000121/finalavyanyselistingstanda.htm).
  - Stockholder approval of amended Articles of Incorporation. [Example](https://www.sec.gov/Archives/edgar/data/320121/000032012120000038/ex99-1.htm)
- Letter to shareholders. [Example](https://www.sec.gov/Archives/edgar/data/1868912/000155837023003002/enfn-20230307xex99d1.htm).
- Conference posters. [Example](https://www.sec.gov/Archives/edgar/data/1430306/000138713123002814/ex99-02.htm).
- Asset acquisition term sheet. [Example](https://www.sec.gov/Archives/edgar/data/1350102/000121390023017935/ea174705ex99-1_ascent.htm).
- Report of independent registered public accounting firm. [Example](https://www.sec.gov/Archives/edgar/data/1464963/000119312523062179/d473294dex991.htm).
- Notification of total voting rights. [Example](https://www.sec.gov/Archives/edgar/data/1832433/000183243323000018/exhibit992monthlytvrannoun.htm).
- Material change reports. [Example](https://www.sec.gov/Archives/edgar/data/1318482/000149315223001126/ex99-2.htm).
- Consulting agreement. [Example](https://www.sec.gov/Archives/edgar/data/1445305/000144530523000004/tromconsultingagreementjan.htm).
- By-laws. [Example](https://www.sec.gov/Archives/edgar/data/1318482/000149315223000426/ex99-2.htm).
- Board actions and resolutions. [Example](https://www.sec.gov/Archives/edgar/data/81157/000165495422016958/pgai_ex991.htm).
- Management resignation letters. [Example](https://www.sec.gov/Archives/edgar/data/1071840/000117184322008186/exh_171.htm).
- Results of annual meeting of stockholders. [Example](https://www.sec.gov/Archives/edgar/data/1326706/000149315222035628/ex99-1.htm).
- Invitations to shareholder meetings. [Example](https://www.sec.gov/Archives/edgar/data/1837248/000183988222029088/ex99-1.htm).
- Earnings call conference slides. [Example](https://www.sec.gov/Archives/edgar/data/1128361/000112836120000036/hope-2020q3earningsconfe.htm).

8-K filings are used to inform investors about a multitude of material events, such as changes to management, board members, auditors, by-laws and more. Each event is categorized under one of 33 items, such as _Item 9.01 Financial Statements and Exhibits_ which covers, among other things, quarterly or annual business performance updates. Refer to [Supported 8-K Section Items](https://sec-api.io/docs/sec-filings-item-extraction-api) for a complete list of all 8-K items.

### Find 8-K Filings with Exhibit 99

An 8-K filing reporting a management change (_Item 5.02 Departure of Directors or Certain Officers_) and one disclosing annual financial results (_Item 2.02 Results of Operations and Financial Condition_ and _Item 9.01 Financial Statements and Exhibits_) can both include Exhibit 99 attachments. For this example, however, only 8-K filings with Exhibit 99 and items 2.02 and 9.01 are relevant.

The search expression criteria are as follows:

- `formType:"8-K"` to include both 8-K and 8-K/A filings,
- `documentFormatFiles.type:99` to focus only on exhibits of type `99` (e.g., `EX-99.1` and `EXHIBIT99`),
- `items:"2.02" AND items:"9.01"` to limit results to filings that include _Item 2.02 Results of Operations and Financial Condition_ and _Item 9.01 Financial Statements and Exhibits_.

Combining these conditions with the `AND` operator results in the following query:

```
formType:"8-K" AND items:"2.02" AND items:"9.01" AND documentFormatFiles.type:(99, 99*, *99, *99*)
```

In [ ]:
search_params = {
    "query": 'formType:"8-K" AND documentFormatFiles.type:(99, 99*, *99, *99*) AND items:"9.01" AND items:"2.02"',
    "from": "0",
    "size": "50",
    "sort": [{"filedAt": {"order": "desc"}}],
}

response = queryApi.get_filings(search_params)

Let's convert the metadata of the first 50 matching filings into a pandas DataFrame.

In [ ]:
import pandas as pd

# convert Query API response into a DataFrame
filings = pd.DataFrame.from_records(response['filings'])

In [ ]:
print('Keys of the metadata for each filing')
print('---------------------------------')
print(*list(filings.keys()), sep='\n')

Keys of the metadata for each filing
---------------------------------
ticker
formType
accessionNo
cik
companyNameLong
companyName
linkToFilingDetails
description
linkToTxt
filedAt
documentFormatFiles
periodOfReport
entities
id
seriesAndClassesContractsInformation
items
linkToHtml
linkToXbrl
dataFiles


In [ ]:
filings.head(3)

,ticker,formType,accessionNo,cik,companyNameLong,companyName,linkToFilingDetails,description,linkToTxt,filedAt,documentFormatFiles,periodOfReport,entities,id,seriesAndClassesContractsInformation,items,linkToHtml,linkToXbrl,dataFiles
0,NUWE,8-K,0001140361-24-045016,1506492,"Nuwellis, Inc. (Filer)","Nuwellis, Inc.",https://www.sec.gov/Archives/edgar/data/150649...,Form 8-K - Current report - Item 2.02 Item 9.01,https://www.sec.gov/Archives/edgar/data/150649...,2024-11-01T16:45:31-04:00,"[{'sequence': '1', 'size': '29515', 'documentU...",2024-11-01,"[{'fiscalYearEnd': '1231', 'stateOfIncorporati...",05cca44e7a981e8c0640dafab4a50149,[],[Item 2.02: Results of Operations and Financia...,https://www.sec.gov/Archives/edgar/data/150649...,,"[{'sequence': '3', 'size': '3997', 'documentUr..."
1,CPSH,8-K,0001437749-24-033004,814676,CPS TECHNOLOGIES CORP/DE/ (Filer),CPS TECHNOLOGIES CORP/DE/,https://www.sec.gov/Archives/edgar/data/814676...,Form 8-K - Current report - Item 2.02 Item 8.0...,https://www.sec.gov/Archives/edgar/data/814676...,2024-11-01T16:40:51-04:00,"[{'sequence': '1', 'size': '24803', 'documentU...",2024-11-01,"[{'fiscalYearEnd': '1228', 'stateOfIncorporati...",aee603751997acf7a312c9543c0e1f53,[],[Item 2.02: Results of Operations and Financia...,https://www.sec.gov/Archives/edgar/data/814676...,,"[{'sequence': '4', 'size': '3409', 'documentUr..."
2,INMB,8-K,0001213900-24-093531,1711754,"Inmune Bio, Inc. (Filer)","Inmune Bio, Inc.",https://www.sec.gov/Archives/edgar/data/171175...,Form 8-K - Current report - Item 2.02 Item 9.01,https://www.sec.gov/Archives/edgar/data/171175...,2024-11-01T16:23:48-04:00,"[{'sequence': '1', 'size': '24650', 'documentU...",2024-10-31,"[{'fiscalYearEnd': '1231', 'stateOfIncorporati...",8a3b08e2f061a3c4d0b97044c089fb76,[],[Item 2.02: Results of Operations and Financia...,https://www.sec.gov/Archives/edgar/data/171175...,,"[{'sequence': '4', 'size': '3018', 'documentUr..."


To extract URLs for all Exhibit 99 files from the `documentFormatFiles` list, iterate through each filing’s metadata and filter the `documentFormatFiles` entries based on the `type` containing "99". Since `type` values are not standardized, partial matching on "99" is necessary to capture variations like `EX-99.1`, `EX-99`, or `EX-99.01`.

In [ ]:
documentFormatFiles = [doc for sublist in list(filings['documentFormatFiles']) for doc in sublist]
exhibit_99s = list(filter(lambda doc: '99' in doc['type'], documentFormatFiles))

In [ ]:
exhibit_99s[:5]

[{'sequence': '2',
  'size': '14187',
  'documentUrl': 'https://www.sec.gov/Archives/edgar/data/1506492/000114036124045016/ef20037953_99-1.htm',
  'description': 'EXHIBIT 99.1',
  'type': 'EX-99.1'},
 {'sequence': '2',
  'size': '200055',
  'documentUrl': 'https://www.sec.gov/Archives/edgar/data/814676/000143774924033004/ex_741572.htm',
  'description': 'EXHIBIT 99.1 PRESS RELEASE',
  'type': 'EX-99.1'},
 {'sequence': '3',
  'size': '184062',
  'documentUrl': 'https://www.sec.gov/Archives/edgar/data/814676/000143774924033004/ex_741707.htm',
  'description': 'EXHIBIT 99.2',
  'type': 'EX-99.2'},
 {'sequence': '2',
  'size': '100966',
  'documentUrl': 'https://www.sec.gov/Archives/edgar/data/1711754/000121390024093531/ea021928902ex99-1_inmune.htm',
  'description': 'PRESS RELEASE OF INMUNE BIO INC., DATED OCTOBER 31, 2024',
  'type': 'EX-99.1'},
 {'sequence': '2',
  'size': '66514',
  'documentUrl': 'https://www.sec.gov/Archives/edgar/data/1538822/000110465924113497/tm2427172d1_ex99-1.ht

With the logic in place to locate metadata for relevant 8-K filings and extract URLs for Exhibit 99 files, the next step is to create a function, `download_metadata(start_year, end_year)`. This function will encapsulate all necessary steps and execute them over a specified range of years. The function will return a DataFrame with the URLs of all Exhibit 99 files from the selected 8-K filings. The results will be saved to a CSV file `exhibit-99-8k-filings-metadata.csv` for further processing.

In [ ]:
from pathlib import Path
import json


def download_metadata(start_year=2020, end_year=2023):
    output_file = "exhibit-99-8k-filings-metadata.csv"
    if Path(output_file).is_file():
        result = pd.read_csv(output_file)
        return result

    print("✅ Starting download process")

    # create ticker batches, with 100 tickers per batch
    frames = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            for from_index in range(0, 9950, 50):
                date_range_query = f"filedAt:[{year}-{month:02d}-01 TO {year}-{month:02d}-31]"
                form_tye_query = 'formType:"8-K"'
                document_format_query = "documentFormatFiles.type:(99, 99*, *99, *99*)"
                items_query = 'items:("9.01" AND "2.02")'
                query = (
                    form_tye_query
                    + " AND "
                    + document_format_query
                    + " AND "
                    + items_query
                    + " AND "
                    + date_range_query
                )

                search_params = {
                    "query": query,
                    "from": from_index,
                    "size": "50",
                    "sort": [{"filedAt": {"order": "desc"}}],
                }

                # print(json.dumps(query))

                response = queryApi.get_filings(search_params)

                if len(response["filings"]) == 0:
                    break

                filings = pd.DataFrame.from_records(response["filings"])

                documentFormatFiles = [
                    doc
                    for sublist in list(filings["documentFormatFiles"])
                    for doc in sublist
                ]

                exhibit_99s_list = list(
                    filter(lambda doc: "99" in doc["type"], documentFormatFiles)
                )

                exhibit_99s_df = pd.DataFrame.from_records(exhibit_99s_list)

                frames.append(exhibit_99s_df)

                print(
                    "Month {year}-{month:02d}, from {from_index} completed".format(
                        year=year, month=month, from_index=from_index
                    )
                )

        print("✅ Downloaded metadata for year", year)

    result = pd.concat(frames)
    result.to_csv(output_file, index=False)

    number_metadata_downloaded = len(result)
    print(
        "✅ Downloaded completed. Metadata downloaded for {} filings.".format(
            number_metadata_downloaded
        )
    )

    return result


exhibit_99s = download_metadata(start_year=2023, end_year=2023)

✅ Starting download process
Month 2023-01, from 0 completed
Month 2023-02, from 0 completed
Month 2023-03, from 0 completed
Month 2023-04, from 0 completed
Month 2023-05, from 0 completed
Month 2023-06, from 0 completed
Month 2023-07, from 0 completed
Month 2023-08, from 0 completed
Month 2023-09, from 0 completed
Month 2023-10, from 0 completed
Month 2023-11, from 0 completed
Month 2023-12, from 0 completed
✅ Downloaded metadata for year 2023
✅ Downloaded completed. Metadata downloaded for 720 filings.


In [ ]:
print('Number of Exhibit 99 URLs found for 2023:', len(exhibit_99s))
exhibit_99s

Number of Exhibit 99 URLs found for 2023: 720


,sequence,size,documentUrl,description,type
0,2,76339,https://www.sec.gov/Archives/edgar/data/117515...,EXHIBIT 99.1,EX-99.1
1,2,310936,https://www.sec.gov/Archives/edgar/data/130221...,EX-99.1,EX-99.1
2,2,4285072,https://www.sec.gov/Archives/edgar/data/103754...,EX-99.1,EX-99.1
3,3,274620,https://www.sec.gov/Archives/edgar/data/103754...,EX-99.2,EX-99.2
4,2,4285072,https://www.sec.gov/Archives/edgar/data/103754...,EX-99.1,EX-99.1
...,...,...,...,...,...
54,3,24419,https://www.sec.gov/Archives/edgar/data/122338...,EX-99.2,EX-99.2
55,2,90940,https://www.sec.gov/Archives/edgar/data/171662...,PRESS RELEASE,EX-99.1
56,4,29311,https://www.sec.gov/Archives/edgar/data/730255...,EX-99.1,EX-99.1
57,2,5475,https://www.sec.gov/Archives/edgar/data/706129...,EX-99.1,EX-99.1


### Download Press Releases from Exhibit 99 as HTML and PDF

With all Exhibit 99 URLs collected, the next step is to download the press releases in both HTML and PDF formats. The Download API retrieves the original HTML content, while the PDF Generator API converts this HTML content into PDF format. The downloaded files are organized using the following folder structure:

```
exhibit-99-files/
    - <cik>/
        - <accessionNo>-<fileName>.htm
        - <accessionNo>-<fileName>.pdf
        - ...
    - ...
```

The `download_exhibit(metadata)` function will handle the folder creation for each company's CIK and download each exhibit into its respective folder. Files are named according to the pattern `<accessionNo>-<fileName>.htm` and `<accessionNo>-<fileName>.pdf`, where `accessionNo` and `fileName` are derived from the metadata.

In [ ]:
from sec_api import RenderApi, PdfGeneratorApi
import os

renderApi = RenderApi(api_key)
pdfGeneratorApi = PdfGeneratorApi(api_key)


def download_exhibit(metadata):
    url = metadata["documentUrl"].replace("ix?doc=/", "")

    try:
        cik = [cik for cik in url.split("/") if cik.isdigit()][0]
        accession_number = [cik for cik in url.split("/") if cik.isdigit()][1]
        new_folder = "./exhibit-99-files/" + cik

        if not os.path.isdir(new_folder):
            os.makedirs(new_folder)

        file_content = renderApi.get_filing(url)
        file_content_pdf = pdfGeneratorApi.get_pdf(url)

        file_name = accession_number + "-" + url.split("/")[-1]
        file_name_pdf = file_name + ".pdf"

        with open(new_folder + "/" + file_name, "w") as f:
            f.write(file_content)

        with open(new_folder + "/" + file_name_pdf, "wb") as f:
            f.write(file_content_pdf)

    except:
        print("❌ download failed: {url}".format(url=url))

To parallelize the exhibit download process with `pandarallel`, configure it to use four worker threads, enabling concurrent downloads of four exhibits at a time. The `download_exhibit` function is then applied to each row of the `exhibit_99s` DataFrame using the `.parallel_apply` method from `pandarallel`.

In [ ]:
!pip install -q pandarallel

In [ ]:
from pandarallel import pandarallel

number_of_workers = 4
pandarallel.initialize(progress_bar=True, nb_workers=number_of_workers, verbose=0)

# run a quick test and download 50 exhibits
sample = exhibit_99s.head(50)
sample.parallel_apply(download_exhibit, axis=1)

# uncomment to download all exhibits
# exhibit_99s.parallel_apply(download_filing, axis=1)

print('✅ Download completed')

✅ Download completed
